[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garrygu/newegg-ai-workshop/blob/main/lv1-beginner-v2/Session%203/Session_3_Train_the_Brain_Student_v2.ipynb)

# 🔍 Session 3 — Train the Brain of Your AI (Student)

Welcome! Today you will teach your AI game how to **recognize pictures**.

By the end of class, you will:
- understand what image classification means
- test an AI model on pictures
- see when AI gets things right and wrong
- save settings for later classes

**Big Story:** We are building one project across three classes.

- Session 3 = build the **brain**
- Session 4 = build the **personality**
- Session 5 = build the **game**

## ⏰ Class Plan (about 2 hours)

1. Warm-up and demo  
2. Load the AI model  
3. Test pictures  
4. Talk about mistakes  
5. Save our labels for next class  
6. Try extension challenges if there is time

In [ ]:
print("Welcome to Session 3! 🧠")
print("Today we train and test the brain of our AI game.")

## Step 1 — Imports
Run this cell first.

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

## Step 2 — Set Your Labels

Change the class names to match your project.

Example:
```python
classes = ["cat", "dog"]
```

In [ ]:
classes = ["cat", "dog"]  # Change these labels if needed
num_classes = len(classes)

print("Labels:", classes)
print("Number of classes:", num_classes)

## Step 3 — Tell the Notebook Where Your Model Is

If your teacher already prepared the model, you may not need to change this.

In [ ]:
model_path = Path("models/classifier.pt")
sample_folder = Path("game_assets")

print("Model path:", model_path)
print("Sample image folder:", sample_folder)

## Step 4 — Build the Model Shape

We are using ResNet18. It is a common image model.

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)

if model_path.exists():
    state = torch.load(model_path, map_location="cpu")
    model.load_state_dict(state)
    print("✅ Model loaded!")
else:
    print("⚠️ Model file not found. The notebook can still run, but predictions will not be meaningful.")

model.eval();

## Step 5 — Create Image Transforms

Transforms help turn pictures into numbers the AI can read.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

## Step 6 — Helper Functions

These functions show an image and let the AI make a guess.

In [ ]:
def show_image(image_path):
    image = Image.open(image_path).convert("RGB")
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis("off")
    plt.show()
    return image

def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")
    x = transform(image).unsqueeze(0)

    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]

    best_idx = int(torch.argmax(probs).item())
    return {
        "label": classes[best_idx],
        "confidence": float(probs[best_idx].item()),
        "all_probs": probs.tolist(),
    }

## Step 7 — Find Some Test Images

Put your pictures inside `game_assets/`.

If there are no images yet, ask your teacher for help.

In [ ]:
image_paths = []
if sample_folder.exists():
    for ext in ["*.png", "*.jpg", "*.jpeg", "*.webp"]:
        image_paths.extend(sample_folder.glob(ext))

image_paths = sorted(image_paths)
print("Found", len(image_paths), "images")

image_paths[:5]

## Step 8 — Test One Image

Pick one image and see what the AI says.

In [ ]:
if image_paths:
    test_image = image_paths[0]
    print("Testing:", test_image.name)
    show_image(test_image)
    result = predict_image(test_image)
    print("AI guess:", result["label"])
    print("Confidence:", round(result["confidence"], 3))
else:
    print("No images found yet.")

## Step 9 — Test Random Images

Run this cell several times.

In [ ]:
if image_paths:
    test_image = random.choice(image_paths)
    print("Testing:", test_image.name)
    show_image(test_image)
    result = predict_image(test_image)
    print("AI guess:", result["label"])
    print("Confidence:", round(result["confidence"], 3))
else:
    print("No images found yet.")

## Step 10 — Compare AI Guess vs Your Guess

Try writing your own guess before reading the AI answer.

In [ ]:
if image_paths:
    test_image = random.choice(image_paths)
    print("Image:", test_image.name)
    show_image(test_image)
    your_guess = input("What do YOU think this is? ")
    result = predict_image(test_image)
    print("Your guess:", your_guess)
    print("AI guess:", result["label"])
    print("Confidence:", round(result["confidence"], 3))
else:
    print("No images found yet.")

## Think Like a Scientist 🧪

Answer these questions in your own words:

1. When does the AI do well?
2. When does the AI make mistakes?
3. Why might the AI be confused?

## Step 11 — Save Settings for Later Classes

We will use these labels again in Sessions 4 and 5.

In [ ]:
settings = {
    "classes": classes,
    "num_classes": num_classes,
    "personality": "friendly"
}

Path("session_data").mkdir(exist_ok=True)

with open("session_data/game_settings.json", "w", encoding="utf-8") as f:
    json.dump(settings, f, indent=2)

print("✅ Saved settings to session_data/game_settings.json")

## Exit Ticket

Before class ends, make sure you can say:

- “My AI brain can look at a picture.”
- “My AI brain can make a guess.”
- “AI can still make mistakes.”

Next class, we will give this AI a **personality**.